Step 1. 데이터 준비 및 정제
•	라벨링 최적화: 7,500장의 정면 사진을 5개 감정(Neutral, Happiness, Anger, Sadness, Panic)으로 재분류.
•	보류 데이터 처리: 모호한 표정들을 별도로 모아 다수결로 재정의하여 모델의 판단 경계 보강.
-> 라벨링 교체 완료
-> BOX이상 데이터 제거 작업 필요함

In [19]:
import pandas as pd
import numpy as np
import os
import json
import zipfile  # 추가
import io       # 추가

In [20]:
# BOX이상 데이터 제거 완료(데이터셋 레벨에서)

In [34]:

# --- [1. 경로 설정] ---
IMG_BASE = '/workspace/데이터셋/img'
# 팀원분이 주신 JSON 파일들이 모여있는 폴더 경로 (현재 작업 디렉토리라면 './')
JSON_DIR = '/workspace/user5/workspace/user5/img/relabeled' 
MODIFIED_NPZ_DIR = '/workspace/user1/data_classification/modified/segmentation'

DATA_TYPES = ['train', 'val']
EMOTIONS = ['anger', 'happiness', 'sadness', 'panic', 'neutral']

# --- [2. 팀원 JSON 통합 함수] ---
def merge_team_jsons():
    print("🔍 팀원 JSON 파일 통합 중...")
    combined_labels = {}
    json_files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json') and ('train_' in f or 'val_' in f)]
    
    for j_file in json_files:
        try:
            with open(os.path.join(JSON_DIR, j_file), 'r', encoding='utf-8-sig') as f:
                data = json.load(f)
            
            images = data.get('images', [])
            for img in images:
                fname = img.get('filename')
                if fname:
                    # 파일명을 키로 해서 bbox와 HSEmotion 정보를 저장
                    combined_labels[fname] = {
                        'bbox': img.get('bbox'),
                        'annot_hs': img.get('annot_HSEmotion', {}),
                        'uploader_exp': img.get('folder_label')
                    }
        except Exception as e:
            print(f"⚠️ {j_file} 로드 실패: {e}")
            
    print(f"✅ 총 {len(combined_labels)}건의 라벨 정보 통합 완료.")
    return combined_labels

# --- [3. 메인 실행 셀] ---
# 1. 팀원 데이터 로드
master_labels = merge_team_jsons()

# 2. 이미지 폴더 전체 스캔 (7,500개 타겟)
all_img_list = []
print("\n🔍 이미지 폴더 전체 스캔 시작...")
for split in DATA_TYPES:
    for emo in EMOTIONS:
        # 폴더명 예외 처리 (happy/happiness 등)
        folder_name = 'happy' if emo == 'happiness' else emo
        emo_path = os.path.join(IMG_BASE, split, folder_name)
        
        if not os.path.exists(emo_path): continue
        
        for fname in os.listdir(emo_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                # 해당 이미지에 대한 라벨 정보가 있는지 확인
                info = master_labels.get(fname, {})
                bbox = info.get('bbox', {})
                annot_hs = info.get('annot_hs', {})
                top1 = annot_hs.get('top1', {})
                
                # 데이터 행 생성
                all_img_list.append({
                    'filename': fname,
                    'data_type': split,
                    'emotion_folder': emo,
                    'img_path': os.path.join(emo_path, fname),
                    # BBox 정보 (마스크 대용)
                    'bbox_minX': bbox.get('minX'),
                    'bbox_minY': bbox.get('minY'),
                    'bbox_maxX': bbox.get('maxX'),
                    'bbox_maxY': bbox.get('maxY'),
                    # HSEmotion 점수 정보
                    'top1_emotion': top1.get('emotion'),
                    'top1_score': top1.get('score'),
                    'has_score': True if top1.get('score') else False,
                    # NPZ 마스크 존재 여부 체크 (나중에 로더에서 활용)
                    'has_npz': os.path.exists(os.path.join(MODIFIED_NPZ_DIR, f"new_combined_{emo}.npz"))
                })

# 3. 데이터프레임 생성 및 저장
df_total = pd.DataFrame(all_img_list)
save_path = '/workspace/user1/model/total_7500_metadata.csv'
df_total.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"\n✨ 작업 완료!")
print(f"📊 총 확보된 이미지: {len(df_total)}개")
print(f"📈 그 중 점수(Score)가 있는 데이터: {df_total['has_score'].sum()}개")
print(f"💾 결과 저장: {save_path}")

🔍 팀원 JSON 파일 통합 중...
✅ 총 0건의 라벨 정보 통합 완료.

🔍 이미지 폴더 전체 스캔 시작...

✨ 작업 완료!
📊 총 확보된 이미지: 7196개
📈 그 중 점수(Score)가 있는 데이터: 0개
💾 결과 저장: /workspace/user1/model/total_7500_metadata.csv


In [37]:
import pandas as pd
import os
import json

# --- [1. 경로 설정] ---
# 팀원분이 주신 label 폴더의 절대 경로
TEAM_LABEL_BASE = '/workspace/user5/workspace/user5/img/relabeled'
# 원본 이미지 경로
IMG_BASE = '/workspace/데이터셋/img'
# 본인이 분류한 NPZ 마스크 경로
MODIFIED_NPZ_DIR = '/workspace/user1/data_classification/modified/segmentation'

# 감정 및 데이터 타입 정의
DATA_TYPES = ['train', 'val']
EMOTIONS = ['anger', 'happiness', 'sadness', 'panic', 'neutral']

# --- [2. 팀원 JSON 통합 함수] ---
def merge_team_jsons_recursive():
    print("🔍 팀원 JSON 파일 통합 시작 (train/val 하위 폴더 스캔)...")
    combined_labels = {}
    file_count = 0
    
    for split in DATA_TYPES:
        # /workspace/user5/workspace/user5/img/relabeled/train 과 같이 경로 생성
        split_path = os.path.join(TEAM_LABEL_BASE, split)
        
        if not os.path.exists(split_path):
            print(f"⚠️ 경로를 찾을 수 없음: {split_path}")
            continue
            
        # 해당 폴더 내의 모든 .json 파일 검색
        json_files = [f for f in os.listdir(split_path) if f.endswith('.json')]
        
        for j_file in json_files:
            full_path = os.path.join(split_path, j_file)
            try:
                with open(full_path, 'r', encoding='utf-8-sig') as f:
                    data = json.load(f)
                
                images = data.get('images', [])
                for img in images:
                    fname = img.get('filename')
                    if fname:
                        combined_labels[fname] = {
                            'bbox': img.get('bbox', {}),
                            'annot_hs': img.get('annot_HSEmotion', {}),
                        }
                file_count += 1
                print(f"✅ 로드 완료: {split}/{j_file} ({len(images)}건)")
            except Exception as e:
                print(f"⚠️ {j_file} 읽기 실패: {e}")
                
    print(f"\n총 {file_count}개 파일에서 {len(combined_labels)}건의 라벨 확보 완료.")
    return combined_labels

# --- [3. 메인 실행 및 CSV 생성] ---
master_labels = merge_team_jsons_recursive()

all_img_list = []
print(f"\n🔍 7,196개 이미지와 마스터 라벨 매칭 중...")

for split in DATA_TYPES:
    for emo in EMOTIONS:
        # 폴더 이름 예외 처리 (happiness -> happy)
        folder_name = 'happy' if emo == 'happiness' else emo
        emo_path = os.path.join(IMG_BASE, split, folder_name)
        
        if not os.path.exists(emo_path): continue
        
        for fname in os.listdir(emo_path):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue
            
            info = master_labels.get(fname, {})
            bbox = info.get('bbox', {})
            annot_hs = info.get('annot_hs', {})
            top1 = annot_hs.get('top1', {})
            
            all_img_list.append({
                'filename': fname,
                'data_type': split,
                'emotion_folder': emo, # 본인의 5분류 기준
                'img_path': os.path.join(emo_path, fname),
                'bbox_minX': bbox.get('minX'),
                'bbox_minY': bbox.get('minY'),
                'bbox_maxX': bbox.get('maxX'),
                'bbox_maxY': bbox.get('maxY'),
                'top1_emotion': top1.get('emotion'),
                'top1_score': top1.get('score'),
                'has_score': True if top1.get('score') else False,
                # 본인이 만든 NPZ 파일 존재 여부
                'has_npz': os.path.exists(os.path.join(MODIFIED_NPZ_DIR, f"new_combined_{emo}.npz"))
            })

df_total = pd.DataFrame(all_img_list)
save_path = '/workspace/user1/model/total_7500_metadata.csv'
df_total.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"\n✨ 최종 통합 결과")
print(f"📊 전체 이미지 대상: {len(df_total)}개")
print(f"📈 점수(Score) 매칭 성공: {df_total['has_score'].sum()}개")
print(f"📍 BBox 좌표 확보: {df_total['bbox_minX'].notnull().sum()}개")
print(f"💾 파일 저장 위치: {save_path}")

🔍 팀원 JSON 파일 통합 시작 (train/val 하위 폴더 스캔)...
✅ 로드 완료: train/train_neutral.json (897건)
⚠️ train_sadness.json 읽기 실패: Expecting property name enclosed in double quotes: line 32903 column 9 (char 786432)
⚠️ train_panic.json 읽기 실패: Expecting property name enclosed in double quotes: line 33014 column 8 (char 786432)
⚠️ train_happy.json 읽기 실패: Expecting ',' delimiter: line 32837 column 28 (char 786432)
⚠️ train_anger.json 읽기 실패: Unterminated string starting at: line 33011 column 11 (char 786431)
✅ 로드 완료: val/val_neutral.json (129건)
✅ 로드 완료: val/val_sadness.json (270건)
✅ 로드 완료: val/val_panic.json (325건)
✅ 로드 완료: val/val_happy.json (289건)
✅ 로드 완료: val/val_anger.json (185건)

총 6개 파일에서 2095건의 라벨 확보 완료.

🔍 7,196개 이미지와 마스터 라벨 매칭 중...



✨ 최종 통합 결과
📊 전체 이미지 대상: 7196개
📈 점수(Score) 매칭 성공: 2093개
📍 BBox 좌표 확보: 2093개
💾 파일 저장 위치: /workspace/user1/model/total_7500_metadata.csv


In [39]:
import pandas as pd
import os
import json
import re

TEAM_LABEL_BASE = '/workspace/user5/workspace/user5/img/relabeled'
IMG_BASE = '/workspace/데이터셋/img'
MODIFIED_NPZ_DIR = '/workspace/user1/data_classification/modified/segmentation'

DATA_TYPES = ['train', 'val']
EMOTIONS = ['anger', 'happiness', 'sadness', 'panic', 'neutral']

def robust_json_loader(file_path):
    """문법 에러가 있는 JSON을 최대한 복구하여 읽어옵니다."""
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            content = f.read()
            
        # 1. 시도: 정상 로드
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            # 2. 시도: 잘린 뒷부분 강제 닫기 로직
            # JSON이 중간에 끊긴 경우 (char 786432 등), 유효한 마지막 이미지까지만 잘라냅니다.
            last_valid_entry = content.rfind('},')
            if last_valid_entry != -1:
                fixed_content = content[:last_valid_entry+1] + "]}" 
                return json.loads(fixed_content)
            return None
    except Exception as e:
        print(f"❌ {os.path.basename(file_path)} 복구 불가: {e}")
        return None

def merge_team_jsons_robust():
    print("🚀 손상된 JSON 강제 복구 및 통합 시작...")
    combined_labels = {}
    
    for split in DATA_TYPES:
        split_path = os.path.join(TEAM_LABEL_BASE, split)
        if not os.path.exists(split_path): continue
            
        for j_file in os.listdir(split_path):
            if not j_file.endswith('.json'): continue
            
            full_path = os.path.join(split_path, j_file)
            data = robust_json_loader(full_path)
            
            if data:
                images = data.get('images', [])
                for img in images:
                    fname = img.get('filename')
                    if fname:
                        combined_labels[fname] = {
                            'bbox': img.get('bbox', {}),
                            'annot_hs': img.get('annot_HSEmotion', {}),
                        }
                print(f"✅ 복구/로드 완료: {j_file} (누적 {len(combined_labels)}건)")
    return combined_labels

# --- 실행 ---
master_labels = merge_team_jsons_robust()

all_img_list = []
for split in DATA_TYPES:
    for emo in EMOTIONS:
        folder_name = 'happy' if emo == 'happiness' else emo
        emo_path = os.path.join(IMG_BASE, split, folder_name)
        if not os.path.exists(emo_path): continue
        
        for fname in os.listdir(emo_path):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue
            info = master_labels.get(fname, {})
            bbox = info.get('bbox', {})
            annot_hs = info.get('annot_hs', {})
            top1 = annot_hs.get('top1', {})
            
            all_img_list.append({
                'filename': fname, 'data_type': split, 'emotion_folder': emo,
                'img_path': os.path.join(emo_path, fname),
                'bbox_minX': bbox.get('minX'), 'bbox_minY': bbox.get('minY'),
                'bbox_maxX': bbox.get('maxX'), 'bbox_maxY': bbox.get('maxY'),
                'top1_score': top1.get('score'),
                'has_score': True if top1.get('score') else False,
                'has_npz': os.path.exists(os.path.join(MODIFIED_NPZ_DIR, f"new_combined_{emo}.npz"))
            })

df_total = pd.DataFrame(all_img_list)
df_total.to_csv('/workspace/user1/model/total_7500_metadata.csv', index=False, encoding='utf-8-sig')

print(f"\n✨ 최종 결과\n📊 전체 대상: {len(df_total)}개\n📈 매칭 성공: {df_total['has_score'].sum()}개")

🚀 손상된 JSON 강제 복구 및 통합 시작...
✅ 복구/로드 완료: train_sadness.json (누적 1609건)
✅ 복구/로드 완료: train_panic.json (누적 3138건)
✅ 복구/로드 완료: train_happy.json (누적 4563건)
✅ 복구/로드 완료: train_blank.json (누적 5123건)
✅ 복구/로드 완료: train_anger.json (누적 6003건)
✅ 복구/로드 완료: val_sadness.json (누적 6273건)
✅ 복구/로드 완료: val_panic.json (누적 6598건)
✅ 복구/로드 완료: val_happy.json (누적 6887건)
✅ 복구/로드 완료: val_blank.json (누적 7016건)
✅ 복구/로드 완료: val_anger.json (누적 7201건)



✨ 최종 결과
📊 전체 대상: 7196개
📈 매칭 성공: 7196개


In [50]:
import os
import numpy as np
import pandas as pd
import json

# 1. 경로 설정
BASE_PATH = '/workspace/데이터셋'
NPZ_BASE_DIR = '/workspace/데이터셋/segmentation'
JSON_BASE_DIR = '/workspace/데이터셋/label'
IMG_BASE_DIR = '/workspace/데이터셋/img'

EMOTIONS = ['anger', 'happy', 'panic', 'sadness']
SPLITS = ['train', 'val']

def generate_final_integrated_metadata():
    all_data = []

    for split in SPLITS:
        for emo in EMOTIONS:
            # A. NPZ 경로 수정: segmentation/{split}/{split}_{emo}.npz
            npz_filename = f"{split}_{emo}.npz"
            npz_path = os.path.join(NPZ_BASE_DIR, split, npz_filename)
            
            npz_filenames = set()
            if os.path.exists(npz_path):
                try:
                    with np.load(npz_path) as data:
                        # 내부 키값 '파일명.jpg.npy'에서 '.npy' 제거
                        npz_filenames = {name.replace('.npy', '') for name in data.files}
                    print(f"📦 [성공] {split}/{npz_filename}: 마스크 {len(npz_filenames)}개")
                except Exception as e:
                    print(f"❌ [에러] {npz_filename} 로드 실패: {e}")
            else:
                # 진단용 출력: 실제로 어디를 찔러봤는지 보여줌
                print(f"⚠️ [미발견] 경로 확인 필요: {npz_path}")

            # B. 이미지 및 JSON 경로 설정
            img_dir = os.path.join(IMG_BASE_DIR, split, emo)
            json_path = os.path.join(JSON_BASE_DIR, split, f"{split}_{emo}.json")
            
            # C. BBox 정보 로드 (cp949 인코딩 유지)
            bbox_lookup = {}
            if os.path.exists(json_path):
                try:
                    with open(json_path, 'r', encoding='cp949') as f:
                        items = json.load(f)
                        for item in items:
                            fname = item.get('filename')
                            # annot_A의 boxes 정보 추출
                            box = item.get('annot_A', {}).get('boxes', {})
                            if fname:
                                bbox_lookup[fname] = box
                except Exception as e:
                    print(f"⚠️ {json_path} 파싱 에러: {e}")

            # D. 최종 매칭 및 리스트업
            if not os.path.exists(img_dir):
                continue
            
            for fname in os.listdir(img_dir):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                
                has_npz = fname in npz_filenames
                bbox = bbox_lookup.get(fname, {})
                
                all_data.append({
                    'filename': fname,
                    'split': split,
                    'emotion': emo,
                    'img_path': os.path.join(img_dir, fname),
                    'has_npz': has_npz,
                    'npz_path': npz_path if has_npz else None,
                    'bbox_minX': bbox.get('minX'),
                    'bbox_minY': bbox.get('minY'),
                    'bbox_maxX': bbox.get('maxX'),
                    'bbox_maxY': bbox.get('maxY')
                })

    # 결과 저장
    df = pd.DataFrame(all_data)
    save_dir = '/workspace/user1/model'
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    save_path = os.path.join(save_dir, 'total_7500_metadata.csv')
    df.to_csv(save_path, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print(f"✅ 통합 완료: 전체 {len(df)}개 이미지")
    print(f"📊 정밀 마스크(NPZ) 매칭 성공: {df['has_npz'].sum()}개")
    print(f"📂 결과 파일: {save_path}")
    print("="*50)

# 실행
generate_final_integrated_metadata()

📦 [성공] train/train_anger.npz: 마스크 1500개
📦 [성공] train/train_happy.npz: 마스크 1495개
📦 [성공] train/train_panic.npz: 마스크 1501개


📦 [성공] train/train_sadness.npz: 마스크 1500개
📦 [성공] val/val_anger.npz: 마스크 300개
📦 [성공] val/val_happy.npz: 마스크 300개
📦 [성공] val/val_panic.npz: 마스크 300개
📦 [성공] val/val_sadness.npz: 마스크 300개

✅ 통합 완료: 전체 7196개 이미지
📊 정밀 마스크(NPZ) 매칭 성공: 7196개
📂 결과 파일: /workspace/user1/model/total_7500_metadata.csv


In [ ]:
# 기본 경로 세팅
ADDRESS = '/workspace/데이터셋'

# img_address = f'{ADDRESS}/img(classify)'
img_address = f'{ADDRESS}/img'

label_address = f'{ADDRESS}/label' # Corrected 'iabel' to 'label'
# label_address = '/workspace/데이터셋/label'
segmentation_address = f'{ADDRESS}/segmentation'
# segmentation_address = f'{ADDRESS}/segmentation'

In [22]:
# csv파일 기반으로 라벨링 재 생성, BOX이상 데이터 제거 완료(데이터셋 레벨에서)
# --- 설정 구간 ---
csv_path = '/workspace/user1/data_classification/3차 프로젝트(4조)-오분류 감정 라벨링_수정판.csv'
save_base_dir = '/workspace/user1/data_classification/modified/label' # 새로운 라벨을 저장할 경로

# 감정 이름 매핑 (CSV의 'happy'를 기존 코드와 맞추기 위해 'happiness'로 변경)
emotion_mapping = {
    'happy': 'happiness',
    'anger': 'anger',
    'sadness': 'sadness',
    'panic': 'panic',
    'neutral': 'neutral'
}

# 1. CSV 데이터 로드 및 정제
try:
    df = pd.read_csv(csv_path)
    print(f"✅ CSV 로드 성공: {len(df)}건")
    
    # 2. 'BOX 이상' 데이터 삭제 (Yes인 행 제거)
    df = df[df['BOX 이상'] != 'Yes'].copy()
    print(f"✂️ 'BOX 이상' 필터링 완료: {len(df)}건 남음")

    df = df.drop_duplicates(subset=['파일명'], keep='last')
    print(f"🧹 파일명 중복 제거 완료: {len(df)}건 남음")

    # 3. 데이터 변환 (컬럼명 변경 및 감정 이름 정규화)
    # 정정 감정 -> uploader_exp / 파일명 -> filename
    df['uploader_exp'] = df['정정 감정'].replace(emotion_mapping)
    # df['uploader_exp'] = df['정정 감정'].map(emotion_mapping)
    kor_mapping = {"기쁨": "happiness", "분노": "anger", "슬픔": "sadness", "당황": "panic", "중립": "neutral"}
    df['uploader_exp'] = df['uploader_exp'].replace(kor_mapping)
    df = df.rename(columns={'파일명': 'filename'})
    
    # 저장 폴더 생성
    os.makedirs(save_base_dir, exist_ok=True)

    # 4. trn/val 및 감정별로 쪼개서 JSON 저장
    splits = df['trn/val'].unique() # ['trn', 'val']
    emotions = df['uploader_exp'].unique()

    for split in splits:
        split_dir = os.path.join(save_base_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        
        # 'trn'은 'train'으로 폴더명/파일명 통일 (기존 코드 호환용)
        type_name = 'train' if split == 'trn' else 'val'
        
        for emo in emotions:
            # 해당 split과 감정에 맞는 데이터 필터링
            subset = df[(df['trn/val'] == split) & (df['uploader_exp'] == emo)]
            
            if subset.empty:
                continue
            
            # 리스트 딕셔너리 형태로 변환
            images_list = subset[['filename', 'uploader_exp']].to_dict(orient='records')
            
            # JSON 구조 생성
            output_json = {"images": images_list}
            
            # 파일 저장 (예: /new_labels/trn/train_anger.json)
            json_filename = f"{type_name}_{emo}.json"
            save_path = os.path.join(split_dir, json_filename)
            
            with open(save_path, 'w', encoding='utf-8-sig') as f:
                json.dump(output_json, f, ensure_ascii=False, indent=4)
                
            print(f"💾 생성 완료: {save_path} ({len(images_list)}건)")

    print("\n✨ 모든 라벨 재생성 작업이 완료되었습니다!")

except Exception as e:
    print(f"❌ 오류 발생: {e}")

✅ CSV 로드 성공: 2119건
✂️ 'BOX 이상' 필터링 완료: 2107건 남음
🧹 파일명 중복 제거 완료: 2084건 남음
💾 생성 완료: /workspace/user1/data_classification/modified/label/trn/train_anger.json (423건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/trn/train_happiness.json (105건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/trn/train_neutral.json (910건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/trn/train_panic.json (111건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/trn/train_sadness.json (216건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/val/val_anger.json (101건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/val/val_happiness.json (14건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/val/val_neutral.json (130건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/val/val_panic.json (51건)
💾 생성 완료: /workspace/user1/data_classification/modified/label/val/val_sadness.json (23건)

✨ 모든 라벨 재생성 작업이 완료되었습니다!


In [23]:

from PIL import Image, ImageOps # Import ImageOps for exif_transpose
import matplotlib.pyplot as plt
import random
from collections import Counter # For mode calculation
from scipy.stats import mode # For mode calculation
import matplotlib.patches as patches # For drawing bounding boxes

# Assuming ADRESS, img_adress, label_adress, segmentation_adress are already defined.

# 이 부분을 건드려서 해당 파트의 파일만 확인 가능
###########################################################################################
EMOTIONS = ['anger', 'happy', 'panic', 'sadness', 'neutral']
# EMOTIONS = ['anger', 'happy', 'panic', 'sadness', 'blank']
DATA_TYPES_TO_PROCESS = ['train', 'val'] # As per the request, focus on 'train' and 'val' for now
###########################################################################################

def load_emotion_data_files(emotion, data_type):
    """
    Loads JSON label data and NPZ segmentation data for a given emotion and data type.
    Returns (json_data_list, npz_data_dict) or (None, None) if files not found.
    """
    json_file_path = os.path.join(label_address, data_type, f'{data_type}_{emotion}.json') # Corrected 'iabel_adress' to 'label_adress'
    npz_file_path = os.path.join(segmentation_address, data_type, f'{data_type}_{emotion}.npz')

    current_json_data = None
    current_npz_data = None

    # Load JSON data (using cp949 encoding as identified in previous steps)
    if os.path.exists(json_file_path):
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                current_json_data = json.load(f)
            print(f"Loaded JSON for {emotion} ({data_type}) from {json_file_path}")
        except Exception as e:
            print(f"Error loading JSON for {emotion} ({data_type}) from {json_file_path}: {e}")
    else:
        print(f"JSON file not found: {json_file_path}")

    # Load NPZ data
    if os.path.exists(npz_file_path):
        try:
            current_npz_data = np.load(npz_file_path)
            print(f"Loaded NPZ for {emotion} ({data_type}) from {npz_file_path}")
        except Exception as e:
            print(f"Error loading NPZ for {emotion} ({data_type}) from {npz_file_path}: {e}")
    else:
        print(f"NPZ file not found: {npz_file_path}")

    return current_json_data, current_npz_data

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

# --- [1. 경로 및 설정] ---
csv_path = '/workspace/user1/data_classification/3차 프로젝트(4조)-오분류 감정 라벨링_수정판.csv'
# 원본 NPZ가 있는 하위 폴더 구조 (train/val)
base_npz_path = '/workspace/데이터셋/segmentation'
# 결과물이 저장될 경로
output_dir = '/workspace/user1/data_classification/modified/segmentation'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 감정 이름 매핑 (CSV 값 -> 시스템에서 사용할 표준 명칭)
emotion_map = {
    'anger': 'anger',
    'happy': 'happiness',
    'sadness': 'sadness',
    'panic': 'panic',
    'neutral': 'neutral'
}

# 저장된 파일 개수를 추적하기 위한 카운터 (메모리를 위해 데이터 자체는 담지 않음)
saved_counts = {emo: 0 for emo in emotion_map.values()}

# --- [2. CSV 데이터 로드 및 필터링] ---
try:
    df = pd.read_csv(csv_path)
    print(f"📊 CSV 로드 완료: {len(df)}건")

    # 'BOX 이상'이 'Yes'인 데이터 제거
    df = df[df['BOX 이상'] != 'Yes'].copy()
    print(f"✂️ 'BOX 이상' 제거 완료: {len(df)}건 남음")

    # [수정됨] 파일명 중복 제거 로직 추가 (Duplicate name 경고 및 인코딩 에러 방지)
    df = df.drop_duplicates(subset=['파일명'], keep='last')
    print(f"🧹 파일명 중복 제거 완료: {len(df)}건 남음")

    # --- [3 & 4. 스트리밍 방식으로 마스크 추출 및 실시간 저장 (OOM 방지)] ---
    print("\n--- 💾 정제된 통합 NPZ 파일 생성 중 (메모리 최적화 모드) ---")

    # 1) 타겟 감정별로 zip(npz) 파일을 미리 열어둠 (np.savez_compressed와 동일한 구조)
    zip_files = {}
    for emo_name in set(emotion_map.values()):
        save_path = os.path.join(output_dir, f'new_combined_{emo_name}.npz')
        zip_files[emo_name] = zipfile.ZipFile(save_path, 'w', compression=zipfile.ZIP_DEFLATED)

    # trn -> train으로 변환하여 경로 매칭
    df['data_type'] = df['trn/val'].apply(lambda x: 'train' if x == 'trn' else 'val')
    
    # 작업 효율을 위해 (데이터타입, 이전감정) 그룹별로 처리
    grouped = df.groupby(['data_type', '이전 감정'])

    for (data_type, source_emo), group_df in grouped:
        # 원본 NPZ 파일명 결정 (예: train_happy.npz)
        npz_filename = f"{data_type}_{source_emo}.npz"
        npz_path = os.path.join(base_npz_path, data_type, npz_filename)

        if not os.path.exists(npz_path):
            print(f"⚠️ NPZ 파일 없음: {npz_path}")
            continue

        print(f"📦 마스크 추출 및 실시간 기록 중: {npz_filename} ({len(group_df)}건 처리 대상)")
        source_npz = np.load(npz_path, allow_pickle=True)

        for _, row in group_df.iterrows():
            filename = row['파일명']
            target_emo_raw = row['정정 감정']
            target_emo = emotion_map.get(target_emo_raw)

            if not target_emo:
                continue

            # NPZ 내의 키값 형식에 맞춰 조회 (.npy 접미사)
            mask_key = filename + '.npy'
            
            if mask_key in source_npz:
                # 마스크 데이터를 메모리에 하나만 올림
                arr = source_npz[mask_key]
                
                # 메모리 버퍼에 NumPy 바이너리 형식으로 변환 후, 열어둔 zip(npz) 파일에 즉시 쓰기
                bytes_io = io.BytesIO()
                np.save(bytes_io, arr)
                zip_files[target_emo].writestr(mask_key, bytes_io.getvalue())
                
                saved_counts[target_emo] += 1
        
        # 해당 그룹의 원본 NPZ 처리가 끝나면 메모리에서 닫기
        source_npz.close()

    # 열어둔 zip 파일들을 모두 안전하게 닫고 결과 출력
    print("\n--- 💾 저장 결과 확인 ---")
    for emo_name, zf in zip_files.items():
        zf.close() # 파일 닫기 완료
        
        count = saved_counts[emo_name]
        save_path = os.path.join(output_dir, f'new_combined_{emo_name}.npz')
        
        if count > 0:
            print(f"✅ 생성 완료: {os.path.basename(save_path)} ({count}건)")
        else:
            print(f"ℹ️ 데이터 없음: {emo_name} (빈 파일 삭제)")
            if os.path.exists(save_path):
                os.remove(save_path) # 데이터가 전혀 없는 경우 생성된 빈 압축파일 정리

    print("\n✨ 모든 전처리가 완료되었습니다. 메모리 초과 및 중복 에러 없이 통합 파일이 생성되었습니다.")

except Exception as e:
    print(f"❌ 작업 중 오류 발생: {e}")

📊 CSV 로드 완료: 2119건
✂️ 'BOX 이상' 제거 완료: 2107건 남음
🧹 파일명 중복 제거 완료: 2084건 남음

--- 💾 정제된 통합 NPZ 파일 생성 중 (메모리 최적화 모드) ---
📦 마스크 추출 및 실시간 기록 중: train_anger.npz (991건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: train_happy.npz (151건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: train_panic.npz (60건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: train_sadness.npz (563건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: val_anger.npz (216건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: val_happy.npz (26건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: val_panic.npz (25건 처리 대상)
📦 마스크 추출 및 실시간 기록 중: val_sadness.npz (52건 처리 대상)

--- 💾 저장 결과 확인 ---
✅ 생성 완료: new_combined_happiness.npz (89건)
✅ 생성 완료: new_combined_panic.npz (139건)
✅ 생성 완료: new_combined_sadness.npz (130건)
✅ 생성 완료: new_combined_neutral.npz (679건)
✅ 생성 완료: new_combined_anger.npz (453건)

✨ 모든 전처리가 완료되었습니다. 메모리 초과 및 중복 에러 없이 통합 파일이 생성되었습니다.


In [33]:
import pandas as pd
import numpy as np
import os
import json
import zipfile

# --- [1. 경로 설정] ---
MODIFIED_BASE = '/workspace/user1/data_classification/modified'
# ORIGINAL_LABEL_BASE = '/workspace/데이터셋/label'
ORIGINAL_LABEL_BASE = '/workspace/user5/workspace/user5/img/relabeled'

# IMG_BASE = '/workspace/데이터셋/img' 
IMG_BASE = '/workspace/user5/workspace/user5/img/img(classify)' 


DATA_TYPES_TO_PROCESS = ['train', 'val']
EMOTIONS = ['anger', 'happiness', 'sadness', 'panic', 'neutral']

def load_original_scores():
    print("🔍 원본 JSON 점수 데이터 복구 및 구조 분석 중...")
    score_dict = {}
    encodings = ['utf-8-sig', 'cp949', 'utf-8']
    
    # 구조 분석을 위한 샘플 저장용
    sample_entry = None
    
    for split in ['train', 'trn', 'val']:
        split_path = os.path.join(ORIGINAL_LABEL_BASE, split)
        if not os.path.exists(split_path): continue
        
        for file in os.listdir(split_path):
            if not file.endswith('.json'): continue
            
            full_path = os.path.join(split_path, file)
            data = None
            for enc in encodings:
                try:
                    with open(full_path, 'r', encoding=enc) as f:
                        data = json.load(f)
                    break
                except:
                    continue
            
            if data is None: continue
            
            images = data.get('images', []) if isinstance(data, dict) else (data if isinstance(data, list) else [])
            
            for item in images:
                if not isinstance(item, dict): continue
                raw_fname = item.get('filename') or item.get('file_name')
                if raw_fname:
                    # [매칭 강화] 경로를 제외한 파일명만 추출하여 소문자 통일
                    fname_only = os.path.basename(str(raw_fname)).strip().lower()
                    if not fname_only.endswith('.jpg'): fname_only += '.jpg'
                    
                    # 'annot_HSEmotion' 키가 없는 경우를 대비해 유연하게 탐색
                    emotion_data = item.get('annot_HSEmotion') or item.get('HSEmotion') or item.get('face_emotion')
                    score_dict[fname_only] = emotion_data if emotion_data else {}
                    
                    if sample_entry is None and emotion_data:
                        sample_entry = (fname_only, emotion_data)
                        
    if sample_entry:
        print(f"💡 데이터 구조 샘플 확인: {sample_entry[0]} -> {list(sample_entry[1].keys())}")
    else:
        print("⚠️ 경고: 원본 JSON에서 'annot_HSEmotion' 또는 유사한 점수 데이터를 찾지 못했습니다.")
        
    print(f"✅ 원본 메타데이터 {len(score_dict)}건 로드 완료")
    return score_dict

def find_original_image_path(img_base, filename):
    # 파일명만 추출
    fname_only = os.path.basename(filename)
    for split in ['train', 'trn', 'val']:
        for emo in ['anger', 'happy', 'happiness', 'sadness', 'panic', 'neutral']:
            test_path = os.path.join(img_base, split, emo, fname_only)
            if os.path.exists(test_path):
                return test_path
    return None

def combined_data_iterator():
    original_scores = load_original_scores()
    
    for data_type in DATA_TYPES_TO_PROCESS:
        for emotion in EMOTIONS:
            label_subfolder = 'trn' if data_type == 'train' else 'val'
            json_filename = f"train_{emotion}.json" if data_type == 'train' else f"val_{emotion}.json"
            
            json_path = os.path.join(MODIFIED_BASE, 'label', label_subfolder, json_filename)
            npz_path = os.path.join(MODIFIED_BASE, 'segmentation', f"new_combined_{emotion}.npz")
            
            if not os.path.exists(json_path) or not os.path.exists(npz_path):
                continue
                
            with open(json_path, 'r', encoding='utf-8-sig') as f:
                json_data = json.load(f)
                
            with zipfile.ZipFile(npz_path, 'r') as npz_segments:
                images_list = json_data.get('images', [])
                npz_file_list = npz_segments.namelist()

                for json_entry in images_list:
                    raw_filename = json_entry.get('filename')
                    if not raw_filename: continue 
                    
                    # 매칭용 파일명 (Basename 기준)
                    fname_only = os.path.basename(str(raw_filename)).strip().lower()
                    if not fname_only.endswith('.jpg'): fname_only += '.jpg'

                    # 점수 데이터 복구 시도
                    annot_hs = original_scores.get(fname_only, {})
                    
                    img_full_path = find_original_image_path(IMG_BASE, raw_filename)
                    mask_key = raw_filename + '.npy'
                    
                    if img_full_path and mask_key in npz_file_list:
                        with npz_segments.open(mask_key) as mask_file:
                            segmentation_mask = np.load(mask_file)
                        
                        top1 = annot_hs.get('top1', {})
                        top2 = annot_hs.get('top2', {})
                        top3 = annot_hs.get('top3', {})
                        all_scores = annot_hs.get('all_scores', {})

                        entry_data = {
                            'filename': raw_filename,
                            'emotion_folder': emotion,
                            'data_type': data_type,
                            'img_path': img_full_path, 
                            'face_detected': annot_hs.get('face_detected'),
                            'top1_emotion': top1.get('emotion'),
                            'top1_score': top1.get('score'),
                            'top2_emotion': top2.get('emotion'),
                            'top2_score': top2.get('score'),
                            'top3_emotion': top3.get('emotion'),
                            'top3_score': top3.get('score'),
                            'uploader_exp': json_entry.get('uploader_exp', emotion), 
                            'segmentation_mask': segmentation_mask
                        }

                        # 8가지 세부 점수 추가
                        if isinstance(all_scores, dict):
                            for emo_key, score_val in all_scores.items():
                                entry_data[f'score_{emo_key.lower()}'] = score_val

                        yield entry_data

# --- [실행 및 저장] ---
print("\n🚀 데이터 통합 및 메타데이터 복구 시작...")
all_matched_data_gen = combined_data_iterator()

metadata_list = []
for entry in all_matched_data_gen:
    meta = {k: v for k, v in entry.items() if k != 'segmentation_mask'}
    metadata_list.append(meta)

df = pd.DataFrame(metadata_list)
csv_save_path = '/workspace/user1/model/step1_metadata.csv' 
os.makedirs(os.path.dirname(csv_save_path), exist_ok=True)

if len(df) > 0:
    df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 완료: 총 {len(df)}건의 데이터가 '{csv_save_path}'에 저장되었습니다.")
    
    # 최종 결과 점검
    fill_rate = df['top1_score'].notnull().mean() * 100
    if fill_rate > 0:
        print(f"🎉 성공! 점수 데이터가 약 {fill_rate:.1f}% 채워졌습니다.")
    else:
        print("⚠️ 주의: 여전히 모든 점수 데이터가 NaN입니다. 원본 JSON 파일 하나를 열어 'annot_HSEmotion' 키가 실제 존재하는지 확인이 필요합니다.")


🚀 데이터 통합 및 메타데이터 복구 시작...
🔍 원본 JSON 점수 데이터 복구 및 구조 분석 중...


💡 데이터 구조 샘플 확인: 007b10b45d424e7876e73b6b66d62d603c6ef2ee886761e570b2110e47608iwlr.jpg -> ['top1', 'top2', 'top3', 'all_scores', 'face_detected']
✅ 원본 메타데이터 2095건 로드 완료

✅ 완료: 총 9건의 데이터가 '/workspace/user1/model/step1_metadata.csv'에 저장되었습니다.
🎉 성공! 점수 데이터가 약 11.1% 채워졌습니다.


In [31]:
import json
import os

# 원본 JSON 파일 중 하나를 샘플로 지정
sample_json_path = '/workspace/데이터셋/label/train/train_anger.json'

# train 폴더에 없으면 trn 폴더에서 탐색
if not os.path.exists(sample_json_path):
    sample_json_path = '/workspace/데이터셋/label/trn/train_anger.json'

try:
    with open(sample_json_path, 'r', encoding='cp949') as f:
        data = json.load(f)
        
    # 데이터 구조 파악
    if isinstance(data, dict) and 'images' in data:
        sample_item = data['images'][0]
    elif isinstance(data, list) and len(data) > 0:
        sample_item = data[0]
    else:
        sample_item = data
        
    print("=== 원본 JSON 첫 번째 데이터 전체 구조 ===")
    print(json.dumps(sample_item, indent=4, ensure_ascii=False))
    
except Exception as e:
    print(f"❌ 파일을 열 수 없습니다: {e}")

=== 원본 JSON 첫 번째 데이터 전체 구조 ===
{
    "filename": "6oj439e3fbcc52759fb3093035b7c0ecc55c93543dae63ec1162f5f6bfc6e481l.jpg",
    "gender": "남",
    "age": 30,
    "isProf": "일반인",
    "faceExp_uploader": "분노",
    "bg_uploader": "교통/이동수단(엘리베이터 포함)",
    "annot_A": {
        "boxes": {
            "maxX": 2073.2546049999996,
            "maxY": 1486.204442,
            "minX": 1338.2201049999999,
            "minY": 539.5066419999999
        },
        "faceExp": "분노",
        "bg": "교통/이동수단(엘리베이터 포함)"
    },
    "annot_B": {
        "boxes": {
            "maxX": 2074.6636049999997,
            "maxY": 1490.044442,
            "minX": 1341.270105,
            "minY": 528.132642
        },
        "faceExp": "분노",
        "bg": "교통/이동수단(엘리베이터 포함)"
    },
    "annot_C": {
        "boxes": {
            "maxX": 2072.188605,
            "maxY": 1487.918442,
            "minX": 1340.0851049999999,
            "minY": 525.650642
        },
        "faceExp": "분노",
        "bg": "교통/이동수단(엘리베이터 포

Step 2. 극장 환경 시뮬레이션 (Augmentation)
•	저조도/저화질 구현: 학습 시 실시간으로 감마 보정(어둡게), 가우시안 노이즈, 블러(뭉개짐)를 적용.
•	강건성 확보: 모델이 깨끗한 사진뿐만 아니라 악조건의 사진에서도 동일한 특징을 추출하도록 유도.


In [ ]:
import cv2
import torch
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

# 1. 극장 환경 증강 파이프라인 (저조도, 노이즈, 블러)
theater_transform = A.Compose([
    A.RandomGamma(gamma_limit=(50, 90), p=0.8), 
    A.GaussNoise(var_limit=(20.0, 60.0), p=0.5), 
    A.Blur(blur_limit=5, p=0.3), 
    A.Resize(224, 224), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 2. Dataset 클래스 정의
class TheaterEmotionDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        
        # HSEmotion 결과값 기준 8개 감정 매핑
        self.label_map = {
            'anger': 0, 'happiness': 1, 'sadness': 2, 'panic': 3, 
            'neutral': 4
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = row['img_path']
        
        # CSV에 추출해둔 top1_emotion을 소문자로 변환하여 라벨 할당
        label_str = str(row['top1_emotion']).lower()
        label = self.label_map.get(label_str, -1) 

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"이미지를 찾을 수 없습니다: {img_path}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

# 3. DataLoader 초기화
train_dataset = TheaterEmotionDataset(
    csv_path='/workspace/user1/model/step1_metadata.csv', 
    transform=theater_transform
)

# VRAM 안정성을 위해 Batch Size는 32로 고정
train_loader = DataLoader(
    train_dataset, 
    batch_size=32,       
    shuffle=True, 
    num_workers=4,       
    pin_memory=True      
)

# 정상 로드 테스트
if __name__ == "__main__":
    print(f"Total Dataset Size: {len(train_dataset)}")
    images, labels = next(iter(train_loader))
    print(f"Batch Image Shape: {images.shape}")
    print(f"Batch Label Shape: {labels.shape}")

Step 3. 이원화 모델 학습 (A40 서버 활용)
•	Model A (CNN 기반): ResNet50 + DAN. 이목구비의 미세한 변화(Attention)를 포착하여 감정 분류.
•	Model B (Transformer 기반): ViT-Base. 이미지 전체의 맥락과 분위기를 파악하여 감정 분류.
•	최적화: 대용량 VRAM(48GB)을 활용해 높은 배치 사이즈와 정규화 전략 적용.
•   Fine tunning작업:Augmentation 된 추가 데이터로 Fine tunning작업. mAP지표 기반 Pretrained model의 성능을 재분배한 라벨링 한 이미지의 방향으로 가중치 변형

Step 4. 감정 데이터 추출 및 비교
•	Top 3 추출: 단순 1등 감정이 아닌, 상위 3개 감정의 이름과 확률값(Confidence)을 리포트.
•	모델 밸런스 평가: 저화질 환경에서 어느 모델이 더 안정적으로 Top 3 분포를 내뱉는지 비교 분석.


Step 5. 감정 합성 (Synthesis) 준비
•	벡터 저장: 각 모델이 뽑아낸 특징 벡터(Feature Vector)와 Top 3 확률을 매칭하여 저장.
•	합성 로직: 추출된 확률 비중을 가중치로 삼아, 생성 모델(GAN/Diffusion)을 통해 새로운 '복합 감정 표정' 생성.
